# 04 - Model comparison and tuning

I compare Ridge, Lasso, random forest and histogram gradient boosting against the linear baseline. Two rules keep this honest. First, every score uses grouped cross-validation, so a model is always judged on sessions it didn't train on. Second, because consecutive rows in a session are nearly identical, I do model selection and tuning on every fifth training row, which is a faithful but much cheaper stand-in for the full set, then refit the chosen model on all the data.

On gradient boosting: scikit-learn's classic `GradientBoostingRegressor` isn't histogram-based and is far too slow at this row count, so I use `HistGradientBoostingRegressor`, the scalable variant, which also handles missing values natively.

In [1]:
import sys
sys.path.append("..")

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_score, GridSearchCV
from sklearn.ensemble import HistGradientBoostingRegressor

from src.data_prep import load_raw, clean, split_by_profile
from src.features import make_preprocessor
from src.model import candidate_models, build_model
from src.evaluate import regression_metrics, score_table

df = clean(load_raw())
X_train, X_test, y_train, y_test, groups_train = split_by_profile(df)

# redundancy-aware subsample for selection and tuning
X_sel, y_sel, g_sel = X_train.iloc[::5], y_train.iloc[::5], groups_train[::5]
print("full train:", len(X_train), "selection subsample:", len(X_sel))

full train: 836542 selection subsample: 167309


## Grouped cross-validation

In [2]:
gkf = GroupKFold(n_splits=4)
rows = []
for name, est in candidate_models().items():
    t = time.time()
    rmse = -cross_val_score(est, X_sel, y_sel, groups=g_sel, cv=gkf,
                            scoring="neg_root_mean_squared_error")
    rows.append({"model": name, "cv_rmse": rmse.mean(), "cv_std": rmse.std(),
                 "seconds": round(time.time() - t)})
pd.DataFrame(rows).set_index("model").round(3)

,cv_rmse,cv_std,seconds
model,,,
ridge,12.833,0.545,0
lasso,12.833,0.545,1
random_forest,13.650,1.284,84
hist_gbm,12.352,1.400,6


The result is not what a leaderboard reflex would expect. Ridge and Lasso land together around 12.8 RMSE; regularisation buys nothing here because there's no real collinearity or high dimensionality to tame. Histogram gradient boosting is the only model that clearly improves on linear. The random forest is actually a touch worse than linear and the most variable across folds.

The random forest story is worth pausing on. It fits the training rows almost perfectly, but each session is a smooth trajectory and the forest memorises those trajectories rather than learning a relationship that transfers to a session it has never seen. Grouped CV exposes that immediately. This is exactly why the validation has to hold out whole sessions: under a naive row-wise split the forest would have looked like the best model by far.

## Tune the gradient boosting model

In [3]:
grid = {
    "model__learning_rate": [0.05, 0.1],
    "model__max_iter": [200, 400],
    "model__max_depth": [None, 8],
}
base = candidate_models()["hist_gbm"]
search = GridSearchCV(base, grid, cv=gkf,
                      scoring="neg_root_mean_squared_error", n_jobs=-1)
search.fit(X_sel, y_sel, groups=g_sel)
print("best params:", search.best_params_)
print("best CV rmse:", round(-search.best_score_, 3))

best params: {'model__learning_rate': 0.05, 'model__max_depth': None, 'model__max_iter': 200}
best CV rmse: 12.207


Tuning helps only a little: a slower learning rate with 200 iterations and unconstrained depth edges the default down to about 12.2 CV RMSE. These settings are what `build_model` uses. The small gain tells me the model was already near the ceiling that this feature set supports, the limiting factor is the information in the seven raw signals, not the boosting hyperparameters.

## Final comparison on the held-out sessions

Each model refit on the full training set, scored on the untouched test sessions. Train and test shown together so overfitting is visible.

In [4]:
final = candidate_models()
final["hist_gbm_tuned"] = build_model()

rows = []
for name, est in final.items():
    est.fit(X_train, y_train)
    tr = regression_metrics(y_train, est.predict(X_train))
    te = regression_metrics(y_test, est.predict(X_test))
    rows.append({"model": name,
                 "train_rmse": tr["rmse"], "test_rmse": te["rmse"],
                 "test_mae": te["mae"], "test_r2": te["r2"]})
pd.DataFrame(rows).set_index("model").round(3)

,train_rmse,test_rmse,test_mae,test_r2
model,,,,
ridge,12.534,11.735,9.204,0.614
lasso,12.534,11.736,9.205,0.614
random_forest,0.596,13.210,10.364,0.511
hist_gbm,5.489,10.846,8.667,0.671
hist_gbm_tuned,5.486,10.843,8.452,0.671


## Decision

The tuned gradient boosting model is the choice: the lowest test RMSE (~10.8 against linear's ~11.7), the highest test R2 (~0.67), and it fits in seconds. Its train-to-test gap is moderate and honest, unlike the random forest, whose near-zero training error and worse-than-linear test error make it a clear example of memorising the training sessions.

So the ranking that matters, on unseen sessions, is gradient boosting first, the linear models close behind, and the random forest last. `build_model` returns this tuned gradient boosting pipeline, and `train.py` will fit it on the full data and save it. The next stage looks at residuals and what the model leans on.